# PRJEB30386 — аудит обрезки V-праймеров

Исходные и очищенные от праймеров записи R1 сопоставляются по SRA ID. Для каждой пары измеряется точный удалённый 5′-префикс и его связь с `v_germline_start`, FWR1 и CDR1 из IgBLAST. Для точного сравнения подстрок Bowtie2 не требуется. Аудит количественно оценивает усечение, но не восстанавливает биологические основания, замещённые PCR-праймером.


In [ ]:
import csv,gzip,json,os,re,statistics
from collections import Counter
from pathlib import Path
VOLUME=Path(os.environ.get("BCR_VOLUME","/data/user/epishkin"))
if not (VOLUME/"results/PRJEB30386").exists():
    found=None
    for start in (Path.cwd().resolve(),Path("/Users/epishkin/workspace/bcr-assembler")):
        for candidate in (start,*start.parents):
            if (candidate/"results/PRJEB30386").exists() and (candidate/"scripts").is_dir(): found=candidate; break
        if found: break
    if not found: raise FileNotFoundError("Cannot locate repository or data root; set BCR_VOLUME")
    VOLUME=found
DATASET="PRJEB30386"; RUNS=["ERR3004229","ERR3004230","ERR3004231","ERR3004232"]
TRIMMED=VOLUME/"results"/DATASET/"trimmed/fastq"; PR=VOLUME/"results"/DATASET/"pr_trimmed/fastq"; AIRR=VOLUME/"results"/DATASET/"annotation/igblast"
OUT=VOLUME/"results"/DATASET/"annotation/qc"; OUT.mkdir(parents=True,exist_ok=True)
print("OUT",OUT)


In [ ]:
def base_id(header):
    x=header.lstrip("@>").split()[0].split("|")[0]
    return re.sub(r"/[12]$","",x)
def fastq_records(path):
    with gzip.open(path,"rt") as h:
        while True:
            head=h.readline()
            if not head:return
            seq=h.readline().rstrip("\n\r"); plus=h.readline(); qual=h.readline()
            if not qual: raise ValueError(f"truncated {path}")
            pm=re.search(r"(?:^|\|)PRIMER=([^|\s]+)",head)
            yield base_id(head),seq,(pm.group(1) if pm else "UNKNOWN")
def p95(values):
    x=sorted(values); return x[min(len(x)-1,int(0.95*len(x)))] if x else None
def cuts_for_run(run):
    wanted={rid:(seq,primer) for rid,seq,primer in fastq_records(PR/f"{run}_1.pr.fastq.gz")}; cuts={}; unmatched=0
    for rid,pre_seq,_ in fastq_records(TRIMMED/f"{run}_1.trim.fastq.gz"):
        item=wanted.get(rid)
        if item is None: continue
        post_seq,primer=item; pos=pre_seq.find(post_seq)
        if pos<0: unmatched+=1
        else: cuts[rid]=(pos,primer)
    return cuts,len(wanted),unmatched
def summarize(group,records):
    cuts=[x["cut_nt"] for x in records]; n=len(records); return {"group":group,"joined_reads":n,"median_cut_nt":statistics.median(cuts) if cuts else None,"p95_cut_nt":p95(cuts),"min_cut_nt":min(cuts) if cuts else None,"max_cut_nt":max(cuts) if cuts else None,"pct_v_germline_start_gt1":100*sum(x["vgs"]>1 for x in records)/n if n else None,"pct_v_germline_start_gt15":100*sum(x["vgs"]>15 for x in records)/n if n else None,"pct_fwr1_present":100*sum(x["fwr1"] for x in records)/n if n else None,"pct_cdr1_present":100*sum(x["cdr1"] for x in records)/n if n else None,"cut_length_histogram":json.dumps(dict(sorted(Counter(cuts).items())))}
def audit_run(run):
    cuts,pr_records,unmatched=cuts_for_run(run); joined=[]
    with open(AIRR/f"{run}.airr.tsv",newline="") as h:
        for r in csv.DictReader(h,delimiter="\t"):
            item=cuts.get(base_id(r["sequence_id"]))
            if item is None: continue
            cut_nt,primer=item
            try:vgs=int(r.get("v_germline_start") or 0)
            except ValueError:vgs=0
            joined.append({"primer":primer,"cut_nt":cut_nt,"vgs":vgs,"fwr1":bool(r.get("fwr1")),"cdr1":bool(r.get("cdr1"))})
    run_row={"run":run,"pr_r1_records":pr_records,"exact_pre_post_matches":len(cuts),"unmatched_substrings":unmatched,**summarize(run,joined)}
    primer_rows=[]
    for primer in sorted({x["primer"] for x in joined}): primer_rows.append({"run":run,"primer":primer,**summarize(primer,[x for x in joined if x["primer"]==primer])})
    return run_row,primer_rows
print("helpers ready")


In [ ]:
run_rows=[]; primer_rows=[]
for run in RUNS:
    rr,pr=audit_run(run); run_rows.append(rr); primer_rows.extend(pr)
for rows,name in [(run_rows,"v_primer_cut_audit_by_run.tsv"),(primer_rows,"v_primer_cut_audit_by_primer.tsv")]:
    out=OUT/name
    with open(out,"w",newline="") as h:
        w=csv.DictWriter(h,fieldnames=list(rows[0]),delimiter="\t"); w.writeheader(); w.writerows(rows)
    print("wrote",out)
print(*run_rows,sep="\n")
